In [ ]:
import pandas as pd 
import numpy as np
import os
import random
import tensorflow as tf
import cv2
from tqdm import tqdm
import datetime
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Concatenate
from tensorflow.keras.layers import Input, Add, Conv2DTranspose
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy, MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.utils import plot_model
from tensorflow.keras import callbacks

from  matplotlib import pyplot as plt
import matplotlib.image as mpimg
from IPython.display import clear_output
%matplotlib inline

from IPython.display import HTML
from base64 import b64encode

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if 'image_2' in root or 'testing' in root:
        print(root)

In [ ]:
print(os.listdir('/kaggle/input'))

In [ ]:
train_data_dir = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/training/image_2/"
train_gt_dir = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/training/gt_image_2/"
test_data_dir = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/testing/"

In [ ]:
TRAINSET_SIZE = int(len(os.listdir(train_data_dir))*0.8)
print(f"Number of Training Examples: {TRAINSET_SIZE}")

VALIDSET_SIZE = int(len(os.listdir(train_data_dir))*0.1)
print(f"NUMBER of Validation Examples: {VALIDSET_SIZE}")

TESTSET_SIZE = int(len(os.listdir(train_data_dir))-TRAINSET_SIZE-VALIDSET_SIZE)
print(f"Number of Testing Examples: {TESTSET_SIZE}")

In [ ]:
IMG_SIZE = 128
N_CHANNELS = 3
N_CLASSES = 1
SEED = 123

In [ ]:
def parse_image(img_path: str) -> dict:
    image = tf.io.read_file(img_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.uint8)

    mask_path = tf.strings.regex_replace(img_path, "image_2", "gt_image_2")
    mask_path = tf.strings.regex_replace(mask_path, "um_", "um_road_")
    mask_path = tf.strings.regex_replace(mask_path, "umm_", "umm_road_")
    mask_path = tf.strings.regex_replace(mask_path, "uu_", "uu_road_")

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=3)

    non_road_label = np.array([255, 0, 0])
    road_label = np.array([255, 0, 255])
    other_road_label = np.array([0, 0, 0])

    mask = tf.experimental.numpy.all(mask == road_label, axis = 2)
    mask = tf.cast(mask, tf.uint8)
    mask = tf.expand_dims(mask, axis=-1)

    return {'image':image, 'segmentation_mask': mask}

In [ ]:
all_dataset = tf.data.Dataset.list_files(train_data_dir + "*.png", shuffle=False)
all_dataset = all_dataset.shuffle(300, seed=SEED, reshuffle_each_iteration=False)
all_dataset = all_dataset.map(parse_image)

train_dataset = all_dataset.take(TRAINSET_SIZE)
val_dataset   = all_dataset.skip(TRAINSET_SIZE).take(VALIDSET_SIZE)
test_dataset  = all_dataset.skip(TRAINSET_SIZE + VALIDSET_SIZE)

In [ ]:
print(len(list(all_dataset)))

In [ ]:
@tf.function
def normalize(input_image: tf.Tensor, input_mask: tf.Tensor) -> tuple:
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask = tf.cast(input_mask, tf.float32)
    return input_image, input_mask

@tf.function
def load_image_train(datapoint: dict) -> tuple:
    input_image = tf.image.resize(datapoint['image'], (IMG_SIZE, IMG_SIZE))
    # PERBAIKAN: method='nearest' supaya mask tetap 0/1
    input_mask = tf.image.resize(datapoint['segmentation_mask'], (IMG_SIZE, IMG_SIZE), method='nearest')

    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        input_mask = tf.image.flip_left_right(input_mask)
    return normalize(input_image, input_mask)

@tf.function
def load_image_test(datapoint: dict) -> tuple:
    input_image = tf.image.resize(datapoint['image'], (IMG_SIZE, IMG_SIZE))
    input_mask = tf.image.resize(datapoint['segmentation_mask'], (IMG_SIZE, IMG_SIZE), method='nearest')
    return normalize(input_image, input_mask)

In [ ]:
# PERBAIKAN: batch lebih kecil, val TIDAK di-repeat
BATCH_SIZE = 8
BUFFER_SIZE = 1000

dataset = {"train": train_dataset, "val": val_dataset, "test": test_dataset}

dataset['train'] = (dataset['train'].map(load_image_train, num_parallel_calls=tf.data.AUTOTUNE)
                    .shuffle(BUFFER_SIZE, seed=SEED).repeat()
                    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
dataset['val']  = dataset['val'].map(load_image_test).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
dataset['test'] = dataset['test'].map(load_image_test).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(dataset['train'])
print(dataset['val'])
print(dataset['test'])

In [ ]:
def display_sample(display_list):
    plt.figure(figsize=(18,18))

    title = ['Input Image', 'True Mask', 'Predicted Mask']

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
        plt.axis('off')

    plt.show()
for image, mask in dataset["train"].take(1):
    sample_image, sample_mask = image, mask

display_sample([sample_image[0], sample_mask[0]])

In [ ]:
vgg16_model = VGG16()
vgg16_model.summary()

In [ ]:
input_shape = (IMG_SIZE, IMG_SIZE, N_CHANNELS)

In [ ]:
inputs = Input(input_shape)
vgg16_model = VGG16(include_top=False, weights='imagenet', input_tensor=inputs)
vgg16_model.trainable = False   # PERBAIKAN: bekukan encoder

c1 = vgg16_model.get_layer("block3_pool").output
c2 = vgg16_model.get_layer("block4_pool").output
c3 = vgg16_model.get_layer("block5_pool").output

u1 = UpSampling2D((2, 2), interpolation='bilinear')(c3)
d1 = Concatenate()([u1, c2])

u2 = UpSampling2D((2, 2), interpolation='bilinear')(d1)
d2 = Concatenate()([u2, c1])
d2 = Conv2D(64, 3, padding='same', activation='relu')(d2)   # PERBAIKAN: tambah conv

u3 = UpSampling2D((8, 8), interpolation='bilinear')(d2)
outputs = Conv2D(N_CLASSES, 1, activation='sigmoid')(u3)

model = Model(inputs, outputs, name="VGG_FCN8")

In [ ]:
# PERBAIKAN: learning rate kecil + metrik IoU biner
model.compile(optimizer=Adam(1e-4),
              loss=BinaryCrossentropy(),
              metrics=[tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou')])

In [ ]:
def create_mask(pred_mask: tf.Tensor) -> tf.Tensor:
    return tf.math.round(pred_mask)

def show_predictions(dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = model.predict(image, verbose=0)
            display_sample([image[0], mask[0], create_mask(pred_mask)[0]])
    else:
        pred = model.predict(sample_image, verbose=0)
        display_sample([sample_image[0], sample_mask[0], create_mask(pred)[0]])

for image, mask in dataset['train'].take(1):
    sample_image, sample_mask = image, mask

show_predictions()

In [ ]:
class DisplayCallback(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)
        show_predictions()
        print('\nSample Prediction after epoch {}\n'.format(epoch+1))

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

callback = [
    DisplayCallback(),
    callbacks.TensorBoard(logdir, histogram_freq = -1),
    callbacks.EarlyStopping(patience = 10, verbose = 1),
    callbacks.ModelCheckpoint('best_model.h5', verbose = 1, save_best_only = True)
]

EPOCHS = 80
STEPS_PER_EPOCH = TRAINSET_SIZE // BATCH_SIZE
VALIDATION_STEPS = VALIDSET_SIZE // BATCH_SIZE

In [ ]:
print(type(callbacks))

In [ ]:
model_history = model.fit(dataset['train'], epochs=EPOCHS,
                          steps_per_epoch=STEPS_PER_EPOCH,
                          validation_data = dataset["val"],
                          validation_steps=VALIDATION_STEPS,
                          callbacks = callback)

In [ ]:
def weighted_img(img, initial_img, α=1., β=0.5, γ=0.):
    return cv2.addWeighted(initial_img, α, img, β, γ)

def process_image_mask(image, mask):
    mask = tf.math.round(mask)
    zero_image = np.zeros_like(mask)
    mask = np.dstack((mask, zero_image, zero_image))
    mask = np.asarray(mask, np.float32)

    image = np.asarray(image, np.float32)

    final_image = weighted_img(mask, image)

    return final_image

In [ ]:
def save_predictions(dataset):
    # Predict and save image the from input dataset
    index = 0
    for batch_image, batch_mask in dataset:
        for image, mask in zip(batch_image, batch_mask):
            print(f"Processing image : {index}")
            pred_mask = model.predict(tf.expand_dims(image, axis = 0))
            save_sample([image, process_image_mask(image, pred_mask[0])], index)
            index += 1

def save_sample(display_list, index):
    plt.figure(figsize=(18, 18))

    title = ['Input Image', 'Predicted Mask']

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
        plt.axis('off')
        
    plt.savefig(f"outputs/{index}.png")
    plt.show()

In [ ]:
os.makedirs("outputs", exist_ok=True)
save_predictions(dataset['test'])